<a href="https://colab.research.google.com/github/vladmsnk/dl_aith/blob/main/contest1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%capture
!pip install torchmetrics

In [ ]:
import os
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

from tqdm.notebook import tqdm
from collections import defaultdict

from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


from IPython.display import clear_output
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme()

from torchmetrics import F1Score



import nltk
from nltk.probability import FreqDist
from nltk.tokenize import word_tokenize

In [ ]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
%%bash
# unzip /content/aith-dl-competition-text-data-2025.zip -d /content/

Archive:  /content/aith-dl-competition-text-data-2025.zip
  inflating: /content/submission.csv  
  inflating: /content/test.csv       
  inflating: /content/train.csv      


In [ ]:
df = pd.read_csv('/content/train.csv')

In [ ]:
df.head()

,id,text,score
0,127857,пидорасы наше правительство издеваються над пе...,1
1,86622,чё цена?,0
2,33294,старый пиндосовский прихвостень...,1
3,421,хорошо потрудились... молодец!!!,0
4,118550,хаба это скучная возня по полу👎,0


In [ ]:
X = df['text']
y = df['score']

In [ ]:
X.head()

,text
0,пидорасы наше правительство издеваються над пе...
1,чё цена?
2,старый пиндосовский прихвостень...
3,хорошо потрудились... молодец!!!
4,хаба это скучная возня по полу👎


In [ ]:
y.head()

,score
0,1
1,0
2,1
3,0
4,0


In [ ]:
%%capture
!pip install pytorch-lightning

In [ ]:
df_test = pd.read_csv('/content/test.csv')

In [ ]:
df_train, df_temp = train_test_split(df, test_size=0.2, random_state=42)
df_val, df_test = train_test_split(df_temp, test_size=0.25, random_state=42)

from nltk.tokenize import TweetTokenizer
from collections import Counter


tokenizer = TweetTokenizer(preserve_case=False, strip_handles=True, reduce_len=True)

all_tokens = []
for text in tqdm(df_train['text'], desc="Tokenizing"):
    if isinstance(text, str):
        tokens = tokenizer.tokenize(text)
        all_tokens.extend(tokens)

max_words = 10000

max_words = 10000

count_tokens = Counter(all_tokens)

most_common = count_tokens.most_common(max_words - 2)
tokens_filtered_top = [word for word, count in most_common]

vocabulary = {word: i for i, word in enumerate(tokens_filtered_top, 2)}

vocabulary["<PAD>"] = 0
vocabulary["<UNK>"] = 1


def text_to_sequence(text, maxlen):
    result = []
    if isinstance(text, str):
        tokens = tokenizer.tokenize(text)

        for word in tokens:
            index = vocabulary.get(word, 1)
            result.append(index)


    if len(result) > maxlen:
        result = result[:maxlen]
    else:
        result = result + [0]*(maxlen-len(result))

    return result

X_train = np.array([text_to_sequence(text, 200) for text in df_train['text']])
X_val = np.array([text_to_sequence(text, 200) for text in df_val['text']])

Tokenizing:   0%|          | 0/139042 [00:00<?, ?it/s]

In [ ]:
import pytorch_lightning as pl
from torch.utils.data import DataLoader, Dataset
import torch
import numpy as np

class TextDataset(Dataset):
    def __init__(self, x, y):
        self.x = torch.from_numpy(x).long()
        self.y = torch.from_numpy(y).float()
    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]


In [ ]:

class ToxicClassificationDataModule(pl.LightningDataModule):
    def __init__(self, df_train, df_val, df_test, max_len=100, batch_size=256):
        super().__init__()
        self.df_train = df_train
        self.df_val = df_val
        self.df_test = df_test
        self.max_len = max_len
        self.batch_size = batch_size

    def setup(self, stage=None):
        if stage == "fit" or stage is None:
            x_train = np.array([text_to_sequence(text, self.max_len)
                               for text in self.df_train["text"]], dtype=np.int32)
            y_train = np.array(self.df_train["score"])

            x_val = np.array([text_to_sequence(text, self.max_len)
                             for text in self.df_val["text"]], dtype=np.int32)
            y_val = np.array(self.df_val["score"])

            self.train_dataset = TextDataset(x_train, y_train)
            self.val_dataset = TextDataset(x_val, y_val)

        if stage == "test" or stage is None:
            x_test = np.array([text_to_sequence(text, self.max_len)
                              for text in self.df_test["text"]], dtype=np.int32)
            y_test = np.array(self.df_test["score"])

            self.test_dataset = TextDataset(x_test, y_test)

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.batch_size)

    def test_dataloader(self):
        return DataLoader(self.test_dataset, batch_size=self.batch_size)

In [ ]:
import pytorch_lightning as pl
import torch
import torch.nn as nn
from torchmetrics import Accuracy, F1Score, AUROC

class BinaryToxicClassifier(pl.LightningModule):
    def __init__(self, vocab_size=10000, embedding_dim=32, out_channel=128, learning_rate=0.001):
        super().__init__()
        self.save_hyperparameters()

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.conv = nn.Conv1d(embedding_dim, out_channel, kernel_size=3)

        self.classifier = nn.Sequential(
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(out_channel, 1)
        )


        self.criterion = nn.BCEWithLogitsLoss()

        self.train_acc = Accuracy(task="binary")
        self.val_acc = Accuracy(task="binary")
        self.train_f1 = F1Score(task="binary")
        self.val_f1 = F1Score(task="binary")

        self.train_auroc = AUROC(task="binary")
        self.val_auroc = AUROC(task="binary")

    def forward(self, x):
        x = self.embedding(x).transpose(1, 2)
        x = self.conv(x)
        x = x.amax(dim=2)
        return self.classifier(x)

    def _shared_step(self, batch, prefix: str):
        x, y = batch
        y = y.float().view(-1, 1)

        logits = self(x)
        loss = self.criterion(logits, y)

        probs = torch.sigmoid(logits)

        preds = (torch.sigmoid(logits) > 0.5).long()
        y_long = y.long()

        if prefix == "train":
            self.train_acc(preds, y_long)
            self.train_f1(preds, y_long)
            self.train_auroc(probs, y_long)

            self.log(f"{prefix}_acc", self.train_acc, prog_bar=True)
            self.log(f"{prefix}_f1", self.train_f1, prog_bar=True)
            self.log(f"{prefix}_auc", self.train_auroc, prog_bar=True)
        else:
            self.val_acc(preds, y_long)
            self.val_f1(preds, y_long)
            self.val_auroc(probs, y_long)

            self.log(f"{prefix}_acc", self.val_acc, prog_bar=True)
            self.log(f"{prefix}_f1", self.val_f1, prog_bar=True)
            self.log(f"{prefix}_auc", self.val_auroc, prog_bar=True)


        self.log(f"{prefix}_loss", loss, prog_bar=True)
        return loss

    def training_step(self, batch, batch_idx):
        return self._shared_step(batch, "train")

    def validation_step(self, batch, batch_idx):
        return self._shared_step(batch, "val")

    def test_step(self, batch, batch_idx):
        return self._shared_step(batch, "test")

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.learning_rate)


In [ ]:
import pytorch_lightning as pl
import torch
import torch.nn as nn
from torchmetrics import AUROC

class TextCNN(pl.LightningModule):
    def __init__(self, vocab_size, embedding_dim=128, num_filters=128, kernel_sizes=(3,4,5),
                 lr=1e-3, dropout=0.5):
        super().__init__()
        self.save_hyperparameters()

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)

        self.convs = nn.ModuleList([
            nn.Sequential(
                nn.Conv1d(embedding_dim, num_filters, k, padding=k//2),
                nn.ReLU(),
                nn.BatchNorm1d(num_filters)
            )
            for k in kernel_sizes
        ])

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(num_filters * len(kernel_sizes), 1)

        self.criterion = nn.BCEWithLogitsLoss()
        self.val_auroc = AUROC(task="binary")
        self.test_auroc = AUROC(task="binary")

    def forward(self, x):
        x = self.embedding(x).transpose(1, 2)
        feats = []
        for conv in self.convs:
            h = conv(x)
            h = h.amax(dim=2)
            feats.append(h)
        h = torch.cat(feats, dim=1)
        h = self.dropout(h)
        return self.fc(h)

    def _step(self, batch, stage: str):
        x, y = batch
        y = y.float().view(-1, 1)

        logits = self(x)
        loss = self.criterion(logits, y)

        probs = torch.sigmoid(logits).view(-1)
        y_long = y.view(-1).long()

        if stage == "val":
            self.val_auroc(probs, y_long)
            self.log("val_auc", self.val_auroc, prog_bar=True)
        elif stage == "test":
            self.test_auroc(probs, y_long)
            self.log("test_auc", self.test_auroc, prog_bar=True)

        self.log(f"{stage}_loss", loss, prog_bar=True)
        return loss

    def training_step(self, batch, batch_idx):
        return self._step(batch, "train")

    def validation_step(self, batch, batch_idx):
        return self._step(batch, "val")

    def test_step(self, batch, batch_idx):
        return self._step(batch, "test")

    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=self.hparams.lr, weight_decay=1e-2)


In [ ]:
import pytorch_lightning as pl
import torch
import torch.nn as nn
from torchmetrics import AUROC

class BiLSTMClassifier(pl.LightningModule):
    def __init__(
        self,
        vocab_size: int,
        embedding_dim: int = 128,
        hidden_size: int = 128,
        num_layers: int = 1,
        lr: float = 2e-3,
        dropout: float = 0.3,
        emb_dropout: float = 0.1,
        bidirectional: bool = True,
        pad_idx: int = 0,
        weight_decay: float = 1e-2,
    ):
        super().__init__()
        self.save_hyperparameters()

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.emb_dropout = nn.Dropout(emb_dropout)

        lstm_dropout = dropout if num_layers > 1 else 0.0
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=bidirectional,
            dropout=lstm_dropout,
        )

        out_dim = hidden_size * (2 if bidirectional else 1)

        self.head = nn.Sequential(
            nn.Linear(out_dim * 2, out_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(out_dim, 1),
        )

        self.criterion = nn.BCEWithLogitsLoss()
        self.val_auroc = AUROC(task="binary")
        self.test_auroc = AUROC(task="binary")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        emb = self.embedding(x)
        emb = self.emb_dropout(emb)

        out, _ = self.lstm(emb)

        mask = (x != self.hparams.pad_idx).unsqueeze(-1)
        lengths = mask.sum(dim=1).clamp(min=1)
        out_mean = (out * mask).sum(dim=1) / lengths

        out_for_max = out.masked_fill(~mask, -1e9)
        out_max = out_for_max.max(dim=1).values

        feats = torch.cat([out_mean, out_max], dim=1)
        logits = self.head(feats)
        return logits

    def _step(self, batch, stage: str):
        x, y = batch
        y = y.float().view(-1, 1)

        logits = self(x)
        loss = self.criterion(logits, y)

        probs = torch.sigmoid(logits).view(-1)
        y_long = y.view(-1).long()

        if stage == "val":
            self.val_auroc(probs, y_long)
            self.log("val_auc", self.val_auroc, prog_bar=True)
        elif stage == "test":
            self.test_auroc(probs, y_long)
            self.log("test_auc", self.test_auroc, prog_bar=True)

        self.log(f"{stage}_loss", loss, prog_bar=True)
        return loss

    def training_step(self, batch, batch_idx):
        return self._step(batch, "train")

    def validation_step(self, batch, batch_idx):
        return self._step(batch, "val")

    def test_step(self, batch, batch_idx):
        return self._step(batch, "test")

    def configure_optimizers(self):
        opt = torch.optim.AdamW(self.parameters(), lr=self.hparams.lr, weight_decay=self.hparams.weight_decay)
        return opt


In [ ]:
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping

data_module = ToxicClassificationDataModule(df_train, df_val, df_test, max_len=100, batch_size=256)



model = BiLSTMClassifier(
    vocab_size=10000,
    embedding_dim=128,
    hidden_size=128,
    num_layers=1,
    lr=2e-3,
    dropout=0.3,
    emb_dropout=0.1,
)


checkpoint_callback = ModelCheckpoint(
    monitor='val_loss',
    mode='min',
    save_top_k=1,
    filename='best-checkpoint'
)

early_stop_callback = EarlyStopping(
    monitor='val_loss',
    patience=5,
    mode='min'
)


trainer = pl.Trainer(
    max_epochs=20,
    callbacks=[early_stop_callback, checkpoint_callback],
    accelerator='gpu',
    log_every_n_steps=10
)

trainer.fit(model, data_module)
trainer.test(model, data_module)

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name        ┃ Type              ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ embedding   │ Embedding         │  1.3 M │ train │     0 │
│ 1 │ emb_dropout │ Dropout           │      0 │ train │     0 │
│ 2 │ lstm        │ LSTM              │  264 K │ train │     0 │
│ 3 │ head        │ Sequential        │  131 K │ train │     0 │
│ 4 │ criterion   │ BCEWithLogitsLoss │      0 │ train │     0 │
│ 5 │ val_auroc   │ BinaryAUROC       │      0 │ train │     0 │
│ 6 │ test_auroc  │ BinaryAUROC       │      0 │ train │     0 │
└───┴─────────────┴───────────────────┴────────┴───────┴───────┘

Trainable params: 1.7 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.7 M                                                                                                
Total estimated model params size (MB): 6                                                                          
Modules in train mode: 11                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_auc          │    0.9713629484176636     │
│         test_loss         │     0.192996546626091     │
└───────────────────────────┴───────────────────────────┘

[{'test_auc': 0.9713629484176636, 'test_loss': 0.192996546626091}]

In [ ]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader

class InferenceTextDataset(Dataset):
    def __init__(self, x: np.ndarray):
        self.x = torch.from_numpy(x).long()

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx]


@torch.no_grad()
def predict_proba_for_df(
    df: pd.DataFrame,
    checkpoint_path: str,
    max_len: int,
    batch_size: int = 512,
    id_col: str = "id",
    text_col: str = "text",
    device: str | None = None,
    output_csv_path: str | None = None,
):
    x = np.array(
        [text_to_sequence(text, max_len) for text in df[text_col]],
        dtype=np.int32
    )
    ds = InferenceTextDataset(x)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False)

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    model = BinaryToxicClassifier.load_from_checkpoint(checkpoint_path)
    model.eval()
    model.to(device)

    probs_all = []
    for xb in dl:
        xb = xb.to(device)
        logits = model(xb)
        probs = torch.sigmoid(logits)
        probs_all.append(probs.squeeze(1).detach().cpu().numpy())

    probs_all = np.concatenate(probs_all, axis=0)

    sub = pd.DataFrame({
        "id": df[id_col].values,
        "score": probs_all.astype(float)
    })

    if output_csv_path is not None:
        sub.to_csv(output_csv_path, index=False)

    return sub



In [ ]:
best_ckpt = checkpoint_callback.best_model_path
print(best_ckpt)

/content/lightning_logs/version_4/checkpoints/best-checkpoint.ckpt


In [ ]:
submission = predict_proba_for_df(
    df=df_test,
    checkpoint_path=best_ckpt,
    max_len=100,
    batch_size=512,
    id_col="id",
    text_col="text",
    output_csv_path="submission.csv"
)

submission.head()

,id,score
0,152591,0.000336
1,108771,0.977476
2,198853,0.031610
3,194736,0.005796
4,88110,0.066959
